# YOLOv11 Training Pipeline — Shelf Void Detection

## Steps:
1. GPU enable karo (Runtime → Change runtime type → T4 GPU)
2. Dataset zip upload karo sidebar 📁 se `/content/` mein
3. Run All
4. `partial.pt` download karo → `backend/models/partial.pt`

In [ ]:
!nvidia-smi
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")
print("Device:", "cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
!pip install -q ultralytics
print("✅ Ultralytics installed")

In [ ]:
import urllib.request
url = "https://raw.githubusercontent.com/your-repo/shelf-void-detection/main/backend/train.py"
# OR upload train.py manually to /content/
import os
if not os.path.exists("/content/train.py"):
    print("Upload train.py to /content/ manually, then re-run this cell.")
    print("Or copy from: https://github.com/your-repo/blob/main/backend/train.py")

### Configuration — Change these as needed

In [ ]:
# === CONFIG ===
ZIP_NAME = "partial_dataset.zip"       # Your dataset zip in /content/
MODEL_NAME = "partial"                  # Model name (partial, arrangement, occupancy)
MODEL_VARIANT = "yolo11s.pt"            # yolo11n.pt (fast) / yolo11s.pt (balanced) / yolo11m.pt (best)
EPOCHS = 120
IMG_SIZE = 960
BATCH = 8
PATIENCE = 35

print(f"Config:")
print(f"  Model: {MODEL_NAME}")
print(f"  Base: {MODEL_VARIANT}")
print(f"  Epochs: {EPOCHS} | Imgsz: {IMG_SIZE} | Batch: {BATCH}")

In [ ]:
import os
import sys
sys.path.insert(0, "/content")

zip_path = f"/content/{ZIP_NAME}"
if not os.path.exists(zip_path):
    print(f"ERROR: {zip_path} not found. Upload your dataset zip first.")
else:
    !python /content/train.py \
        --zip "{zip_path}" \
        --name {MODEL_NAME} \
        --model {MODEL_VARIANT} \
        --epochs {EPOCHS} \
        --imgsz {IMG_SIZE} \
        --batch {BATCH} \
        --patience {PATIENCE} \
        --output /content/runs
    print("✅ Training complete")

In [ ]:
from google.colab import files
import os

best_path = f"/content/runs/{MODEL_NAME}/weights/best.pt"
if os.path.exists(best_path):
    files.download(best_path)
    size = os.path.getsize(best_path) / 1024 / 1024
    print(f"✅ Downloading... ({size:.1f} MB)")
    print(f"   Rename to {MODEL_NAME}.pt and copy to backend/models/")
else:
    print(f"ERROR: {best_path} not found")

In [ ]:
from IPython.display import Image, display
run_dir = f"/content/runs/{MODEL_NAME}"
if os.path.exists(run_dir):
    for img_name in ["results.png", "confusion_matrix.png", "val_batch0_pred.jpg"]:
        p = f"{run_dir}/{img_name}"
        if os.path.exists(p):
            print(f"--- {img_name} ---")
            display(Image(filename=p, width=600))